# 03 · Inverse kinematics of the Franka FR3

The FR3 has **7 joints** but we only ask for **3 coordinates**: every reachable point has *infinitely many* solutions.
We compare the **naive** regression with the **distal teacher**, as in the 2-link warm-up.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from robot_learning_course import fr3
from robot_learning_course.paths import DATA_DIR, MODELS_DIR, find_file
from robot_learning_course.learning.data import load_dataset, split_indices
from robot_learning_course.learning.models import load_model
from robot_learning_course.kinematics import TrueKinematics

q, x = load_dataset(os.path.join(DATA_DIR, "fk_dataset.npz"))
train, val, test = split_indices(len(q))
true_kin = TrueKinematics()

## 1. How different are the solutions?
Pick a target and collect all dataset samples that end within 3 cm of it.

In [ ]:
x_star = np.array([0.45, 0.0, 0.45])
near = np.linalg.norm(x.numpy() - x_star, axis=1) < 0.03
q_near = q.numpy()[near]
print(f"{near.sum()} samples reach x* (within 3 cm)")
q_avg = q_near.mean(0)
print("std of each joint among them [rad]:", q_near.std(0).round(2))
print(f"the AVERAGE configuration reaches a point {1000 * np.linalg.norm(true_kin.fk(q_avg) - x_star):.0f} mm away from x*")

plt.figure(figsize=(8, 3.5))
for j in range(7):
    plt.scatter(np.full(len(q_near), j + 1) + np.random.uniform(-0.15, 0.15, len(q_near)), q_near[:, j], s=6, alpha=0.5)
plt.plot(range(1, 8), q_avg, "k_", ms=30, mew=3, label="average")
plt.xlabel("joint"); plt.ylabel("angle [rad]"); plt.title("configurations reaching the same point"); plt.legend(); plt.show()

## 2. Train the two inverse models
In a terminal (each takes ~1 minute):
```bash
train_ik --mode naive
train_ik --mode distal
```
Pretrained models are used if you did not train them.

In [ ]:
ik_naive = load_model(find_file(os.path.join(MODELS_DIR, "ik_naive.pt")))
ik_distal = load_model(find_file(os.path.join(MODELS_DIR, "ik_distal.pt")))

def reach_mm(ik, targets):
    with torch.no_grad():
        q_pred = ik(targets).numpy()
    return 1000 * np.linalg.norm(true_kin.fk(q_pred) - targets.numpy(), axis=1), q_pred

targets = x[test][:3000]
e_naive, _ = reach_mm(ik_naive, targets)
e_distal, q_distal = reach_mm(ik_distal, targets)
print(f"naive : mean {e_naive.mean():6.1f} mm   median {np.median(e_naive):6.1f} mm")
print(f"distal: mean {e_distal.mean():6.1f} mm   median {np.median(e_distal):6.1f} mm")
bins = np.logspace(0, 3.3, 60)
plt.hist(e_naive, bins=bins, alpha=0.6, label="naive"); plt.hist(e_distal, bins=bins, alpha=0.6, label="distal teacher")
plt.xscale("log"); plt.xlabel("reaching error with the TRUE kinematics [mm]"); plt.legend(); plt.show()

## 3. Where does the distal IK fail?
Colour each target by its error: are errors random, or concentrated somewhere?

In [ ]:
t = targets.numpy()
r = np.linalg.norm(t[:, :2], axis=1)            # horizontal distance from the base
sc = plt.scatter(r, t[:, 2], c=np.clip(e_distal, 0, 60), s=4, cmap="viridis")
plt.colorbar(sc, label="error [mm] (clipped at 60)")
plt.xlabel("horizontal distance from the base [m]"); plt.ylabel("z [m]"); plt.axis("equal"); plt.show()

❓ *Where are the largest errors? (hint: workspace boundary, close to the base, near joint limits).*
Remember: the distal teacher trusts the learned FK. Where $f_\theta$ is wrong, $g_\phi$ is wrong too.

## 4. Smoothness: follow a circle with the IK network
For control we need **neighbouring targets → neighbouring configurations**. Does the network give that?

In [ ]:
s = np.linspace(0, 2 * np.pi, 400)
circle = torch.tensor(np.stack([np.full_like(s, 0.45), 0.12 * np.sin(s), 0.45 + 0.12 * np.cos(s)], 1), dtype=torch.float32)
with torch.no_grad():
    q_circle = ik_distal(circle).numpy()
plt.plot(s, q_circle); plt.xlabel("position along the circle [rad]"); plt.ylabel("joint angles [rad]")
plt.legend([f"q{j+1}" for j in range(7)], ncol=4, fontsize=8); plt.show()
err_c = 1000 * np.linalg.norm(true_kin.fk(q_circle) - circle.numpy(), axis=1)
print(f"circle: mean reaching error {err_c.mean():.1f} mm")

**Take-home:** the inverse map is *one* choice among infinitely many. Learning it requires choosing *how* to break the tie
(here: "reach it, and stay close to home"). In the next step we avoid learning the inverse at all:
we will control the robot **through the Jacobian of the forward model**, with feedback from the sensors.

## ✍️ Optional exercises
* Change `--beta` in `train_ik` (e.g. 0, 1e-2, 1e-1). How do the reaching error and the smoothness of the circle change?
* Is `q_circle` continuous? What would happen if you sent these joint angles directly to the robot, one after the other?